# AlloyTower Property Price Prediction
Train Ridge Regression, Random Forest, and XGBoost with MLflow tracking and SHAP explanations.

In [1]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import shap
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, median_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
load_dotenv('../.env')
print('Imports OK')

c:\Users\bubut\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
mlflow_uri = os.getenv('MLFLOW_TRACKING_URI', 'sqlite:///mlflow.db')
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment('alloytower-predictions')
print(f'MLflow tracking URI: {mlflow_uri}')

2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/04/25 21:01:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/04/25 21:01:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/25 21:01:38 INFO mlflow.store.db.utils: Updating database tables
2026/04/25 21:01:38 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/04/25 21:01:38 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/04/25 21:01:38 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/04/25 21:0

MLflow tracking URI: sqlite:///mlflow.db


In [3]:
df = pd.read_csv('../data/processed/ft_eng.csv')
print(f'Loaded processed data: {df.shape}')
print(df.dtypes)
df.head(3)

Loaded processed data: (2090, 17)
city                           float64
zip_code                         int64
latitude                       float64
longitude                      float64
bedrooms                         int64
bathrooms                      float64
sqft                             int64
lot_size_sqft                    int64
year_built                       int64
last_sale_price                float64
days_since_sale                  int64
tax_year                         int64
owner_occupied                   int64
property_type_Condo              int64
property_type_Multi Family       int64
property_type_Single Family      int64
property_type_Townhouse          int64
dtype: object


,city,zip_code,latitude,longitude,bedrooms,bathrooms,sqft,lot_size_sqft,year_built,last_sale_price,days_since_sale,tax_year,owner_occupied,property_type_Condo,property_type_Multi Family,property_type_Single Family,property_type_Townhouse
0,1.653053e+06,46048,27.489634,-144.860200,6,5.0,1312,10986,1974,546937.17,384,2024,0,0,0,1,0
1,4.970262e+05,87236,28.372054,-81.370872,6,4.0,3387,7939,1939,287944.73,3128,2022,0,0,0,1,0
2,5.114324e+05,59615,23.088205,-131.493490,7,3.5,5329,8479,2010,174253.02,2709,2023,1,0,0,0,1


In [4]:
with open('../models/feature_columns.json') as f:
    feature_columns = json.load(f)

print(f'Feature columns ({len(feature_columns)}):')
print(feature_columns)

Feature columns (16):
['city', 'zip_code', 'latitude', 'longitude', 'bedrooms', 'bathrooms', 'sqft', 'lot_size_sqft', 'year_built', 'days_since_sale', 'tax_year', 'owner_occupied', 'property_type_Condo', 'property_type_Multi Family', 'property_type_Single Family', 'property_type_Townhouse']


In [5]:
X = df[feature_columns]
y = df['last_sale_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Target mean  train={y_train.mean():,.0f}  test={y_test.mean():,.0f}')

Train: (1672, 16)  Test: (418, 16)
Target mean  train=786,757  test=762,938


In [6]:
def evaluate(model, X_test, y_test):
    preds = model.predict(X_test)
    r2   = r2_score(y_test, preds)
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae  = float(median_absolute_error(y_test, preds))
    return {'R2': round(r2, 4), 'RMSE': round(rmse, 2), 'MedAE': round(mae, 2)}


def save_shap_plot(explainer, X_sample, model_name):
    shap_vals = explainer(X_sample)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_vals, X_sample, show=False)
    plt.tight_layout()
    fname = f'shap_summary_{model_name}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact(fname)
    os.remove(fname)
    print(f'  SHAP plot logged: {fname}')


print('Helper functions defined')

Helper functions defined


## 1. Random Forest

In [7]:
param_dist_rf = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [None, 10, 20, 30],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', 0.5],
}

with mlflow.start_run(run_name='RandomForest'):
    rs_rf = RandomizedSearchCV(
        RandomForestRegressor(random_state=42),
        param_dist_rf,
        n_iter=50, cv=5, scoring='r2',
        random_state=42, n_jobs=-1
    )
    rs_rf.fit(X_train, y_train)
    best_rf = rs_rf.best_estimator_

    rf_metrics = evaluate(best_rf, X_test, y_test)

    mlflow.log_params(rs_rf.best_params_)
    mlflow.log_metrics(rf_metrics)

    # SHAP — TreeExplainer (sample 200 rows for speed)
    X_shap = X_test.sample(min(200, len(X_test)), random_state=42)
    tree_explainer_rf = shap.TreeExplainer(best_rf)
    shap_vals_rf = tree_explainer_rf.shap_values(X_shap)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_vals_rf, X_shap, show=False)
    plt.tight_layout()
    plt.savefig('shap_summary_RandomForest.png', dpi=150, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact('shap_summary_RandomForest.png')
    os.remove('shap_summary_RandomForest.png')

    joblib.dump(best_rf, '../models/rf_model.pkl')
    mlflow.sklearn.log_model(best_rf, 'rf_model')

    print(f'RF best params: {rs_rf.best_params_}')
    print(f'RF metrics: {rf_metrics}')

2026/04/25 21:02:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


RF best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 30}
RF metrics: {'R2': 0.5852, 'RMSE': 402820.93, 'MedAE': 202520.85}


## 2. XGBoost

In [8]:
param_dist_xgb = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [3, 5, 7, 9],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'subsample':         [0.6, 0.8, 1.0],
    'colsample_bytree':  [0.6, 0.8, 1.0],
    'gamma':             [0, 0.1, 0.2],
    'reg_alpha':         [0, 0.1, 1.0],
    'reg_lambda':        [1, 1.5, 2],
}

with mlflow.start_run(run_name='XGBoost'):
    rs_xgb = RandomizedSearchCV(
        XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
        param_dist_xgb,
        n_iter=50, cv=5, scoring='r2',
        random_state=42, n_jobs=1
    )
    rs_xgb.fit(X_train, y_train)
    best_xgb = rs_xgb.best_estimator_

    xgb_metrics = evaluate(best_xgb, X_test, y_test)

    mlflow.log_params(rs_xgb.best_params_)
    mlflow.log_metrics(xgb_metrics)

    # SHAP — TreeExplainer
    tree_explainer_xgb = shap.TreeExplainer(best_xgb)
    shap_vals_xgb = tree_explainer_xgb.shap_values(X_test)
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_vals_xgb, X_test, show=False)
    plt.tight_layout()
    plt.savefig('shap_summary_XGBoost.png', dpi=150, bbox_inches='tight')
    plt.close()
    mlflow.log_artifact('shap_summary_XGBoost.png')
    os.remove('shap_summary_XGBoost.png')

    joblib.dump(best_xgb, '../models/gb_model.pkl')
    mlflow.sklearn.log_model(best_xgb, 'gb_model')

    print(f'XGBoost best params: {rs_xgb.best_params_}')
    print(f'XGBoost metrics: {xgb_metrics}')

2026/04/25 21:04:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost best params: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.8}
XGBoost metrics: {'R2': 0.5835, 'RMSE': 403639.21, 'MedAE': 209293.8}


In [9]:
with open('../models/feature_columns.json', 'w') as f:
    json.dump(feature_columns, f, indent=2)
print('feature_columns.json saved (confirmed after training)')

feature_columns.json saved (confirmed after training)


## Model Comparison

In [11]:
results = pd.DataFrame(
    {
        'RandomForest': rf_metrics,
        'XGBoost':      xgb_metrics,
    }
).T

results = results.sort_values('R2', ascending=False)
print('\n=== Model Comparison (test set) ===')
print(results.to_string())
print()
print(f'Best model by R²: {results.index[0]}')
print()
print('To switch the deployed model, update BEST_MODEL in .env:')
print('  BEST_MODEL=models/gb_model.pkl     # XGBoost')
print('  BEST_MODEL=models/rf_model.pkl     # Random Forest')


=== Model Comparison (test set) ===
                  R2       RMSE      MedAE
RandomForest  0.5852  402820.93  202520.85
XGBoost       0.5835  403639.21  209293.80

Best model by R²: RandomForest

To switch the deployed model, update BEST_MODEL in .env:
  BEST_MODEL=models/gb_model.pkl     # XGBoost
  BEST_MODEL=models/rf_model.pkl     # Random Forest
